# Jupyter Test - Packages

Lightweight checks for installed packages: imports, versions, small computations, plotting, and external tool availability (octave, sage, inkscape). Run the notebook top-down.

In [ ]:
print('Python executable:', __import__('sys').executable)
packages = [
    'numpy','pandas','xarray','scipy','matplotlib','sympy','sklearn',
    'autograd','jax','tensorflow','torch','keras','seaborn'
]
for p in packages:
    try:
        m = __import__(p)
        ver = getattr(m, '__version__', getattr(m, 'version', 'unknown'))
        print(f"{p}: OK, version={ver}")
    except Exception as e:
        print(f"{p}: ERROR -> {e}")

In [ ]:
# Numeric / array / dataframe / xarray / scipy quick tests
import numpy as np
import pandas as pd
import xarray as xr
from scipy import linalg

A = np.array([[7,8,8],[9,9,1],[8,4,4]])
print('A:\n', A)
print('det(A) via scipy:', linalg.det(A))

df = pd.DataFrame({'a':[1,2,3],'b':[4,5,6]})
print('pandas df:\n', df)

arr = xr.DataArray(np.random.rand(2,3), dims=('x','y'), coords={'x':[0,1],'y':[0,1,2]})
print('xarray DataArray shape:', arr.shape)

In [ ]:
# Matplotlib plotting and file export (PNG and SVG). Attempt inkscape->PDF if available.
import matplotlib.pyplot as plt
import os, shutil, subprocess

plt.figure()
plt.plot([0,1,2,3],[0,1,4,9], marker='o')
plt.title('Test plot')
png = 'test_plot.png'
svg = 'test_plot.svg'
pdf = 'test_plot.pdf'
plt.savefig(png)
plt.savefig(svg)
plt.close()
print('Saved', png, 'and', svg)

if shutil.which('inkscape'):
    try:
        subprocess.run(['inkscape', svg, '--export-type=pdf', '-o', pdf], check=True)
        print('Converted SVG -> PDF using inkscape:', pdf)
    except Exception as e:
        print('Inkscape conversion failed:', e)
else:
    print('Inkscape not found; skipping SVG->PDF conversion')

In [ ]:
# Autograd and JAX simple gradient tests (if installed)
try:
    from autograd import grad
    import autograd.numpy as anp
    f = lambda x: anp.sin(x) + x**2
    g = grad(f)
    print('autograd grad(1.0)=', g(1.0))
except Exception as e:
    print('autograd not available or failed:', e)

try:
    import jax, jax.numpy as jnp
    jres = jax.grad(lambda x: jnp.sin(x) + x**2)(1.0)
    print('jax version', jax.__version__, 'grad(1.0)=', float(jres))
except Exception as e:
    print('jax not available or failed:', e)

In [ ]:
# TensorFlow / Keras / PyTorch basic checks
try:
    import tensorflow as tf
    print('tensorflow', tf.__version__)
    print('tf simple op:', tf.reduce_sum(tf.constant([1,2,3])).numpy())
except Exception as e:
    print('tensorflow not available or failed:', e)

try:
    # Keras may exist as standalone or as tf.keras
    try:
        import keras
        print('keras (standalone)', keras.__version__)
    except Exception:
        from tensorflow import keras
        print('keras via tensorflow', keras.__version__)
except Exception as e:
    print('keras not available or failed:', e)

try:
    import torch
    print('torch', torch.__version__, 'cuda_available=', torch.cuda.is_available())
except Exception as e:
    print('torch not available or failed:', e)

In [ ]:
# Octave and Sage smoke tests via subprocess
import shutil, subprocess

if shutil.which('octave'):
    try:
        r = subprocess.run(['octave', '--eval', "disp(available_graphics_toolkits());"], capture_output=True, text=True)
        print('octave available, sample output:\n', r.stdout[:500])
    except Exception as e:
        print('octave execution failed:', e)
else:
    print('octave not found in PATH')

if shutil.which('sage'):
    try:
        r = subprocess.run(['sage', '-c', 'print(2+3)'], capture_output=True, text=True)
        print('sage output:', r.stdout.strip())
    except Exception as e:
        print('sage execution failed:', e)
else:
    print('sage not found in PATH')

In [ ]:
# Final environment check
import sys, platform
print('Python', sys.version.splitlines()[0])
print('Platform:', platform.platform())
print('Files saved in notebook working dir:', [f for f in ['test_plot.png','test_plot.svg','test_plot.pdf'] if __import__('os').path.exists(f)])